# 365 Probabilidades - Dia #118
## Qual a probabilidade de a previsão do tempo estar exagerando a chuva?

**Tipo:** Preditivo
**Data de publicação:** 2026-10-09
**Ferramenta:** Python
**Decisão analisada:** Quanto desconto dou ao número da previsão antes de sair para treinar?
**Hashtag:** #365Probabilidades #Dia118

---

### 📖 A História

Foram vários dias seguidos de alerta de tempestade no celular. Antes de cada treino eu
olho a previsão e decido se saio.

A previsão vem em forma de probabilidade, e probabilidade é o assunto deste projeto.
Quis saber quanto dá para confiar nesse número.

---

### 📚 O Conceito: o que é acertar uma probabilidade

Uma previsão de 20% de chuva não erra quando chove nem quando faz sol. Ela só pode ser
julgada em conjunto. Se em todos os dias em que a previsão disse 20% choveu em cerca de
20% deles, ela está calibrada.

Dá também para julgar do jeito que as pessoas usam a previsão. Abaixo de 50% eu entendo
"não vai chover", de 50% para cima eu entendo "vai chover", e conto quantas vezes isso
se confirmou.

A primeira régua é a que responde à pergunta de hoje. Se a previsão anuncia mais chuva
do que cai, ela está exagerando. A segunda entra no fim, como apoio.

O estudo usado aqui conferiu as previsões de probabilidade de chuva do The Weather
Channel, nos Estados Unidos, contra a chuva medida em estações oficiais. A janela é das
7h às 19h, e conta como chuva qualquer registro a partir de 0,01 polegada.

O alerta de tempestade é outro tipo de aviso. O INMET emite avisos de risco por região,
com a lista dos municípios incluídos. O aviso amarelo de tempestade fala em chuva entre
20 e 30 mm por hora ou até 50 mm por dia. Ele não traz probabilidade, então não dá para
medir calibração. Dá para contar em quantos dias com aviso choveu.

---

### 🧮 O Modelo

Cinco camadas.

1. **O exagero medido.** A média do que a previsão anunciou contra o que choveu, e a
   posterior de Jeffreys sobre a frequência de chuva nos dias em que a previsão dizia
   20%, com teste de sensibilidade para previsões que não são independentes entre si.
2. **Cem dias de 20%.** Simulação de Monte Carlo de quantos dias de chuva aparecem em
   cem dias com essa previsão.
3. **A antecedência.** O ganho da previsão sobre a média histórica para cada dia de
   antecedência, de 0 a 9, com uma reta ajustada.
4. **E ela acerta?** Reconstrução da taxa de acerto de chuva ou não chuva, com corte em
   50%, a partir das contagens publicadas. A comparação é com quem dissesse "não vai
   chover" todos os dias.
5. **O dado brasileiro.** Os avisos de chuva do INMET que incluíam Bauru, cruzados com a
   chuva medida na estação automática da cidade. A conta é a proporção de dias com
   chuva entre os dias com aviso e entre os dias sem aviso, com posterior de Jeffreys.

**Fontes:**
- Bickel, J. E. & Kim, S. D. (2008). Verification of The Weather Channel Probability of
  Precipitation Forecasts. *Monthly Weather Review*, 136(12), 4867-4881.
  N = 169.163 previsões, 42 cidades dos Estados Unidos, de 02/11/2004 a 16/01/2006.
  Para o mesmo dia, N = 17.338. Verificação de previsão contra chuva observada.
- INMET, Instituto Nacional de Meteorologia. Avisos meteorológicos (Alert-AS), consultados
  um a um em apiprevmet3.inmet.gov.br/avisos/rss/ID, do ID 50000 ao 55931. Foram lidos
  5.927 avisos emitidos para o Brasil desde 28/02/2025, e 5 IDs retornaram erro do
  servidor. Usei os avisos de Chuvas Intensas, Tempestade e Acumulado de Chuva que
  listavam Bauru (código IBGE 3506003) e não constavam como cancelados.
- INMET. Dados históricos da estação automática A705 (Bauru), chuva horária, arquivos
  anuais de 2025 e 2026 em portal.inmet.gov.br/dadoshistoricos. N = 392 dias com medição
  válida, entre 01/03/2025 e 31/08/2026. Consulta em 03/10/2026.

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. Ele vale para proporções
de pesquisa de opinião autorrelatada, e aqui a medida é chuva registrada em estação.

**Nota de autoria.** O Bickel deste artigo é J. Eric Bickel. Não é o mesmo autor do #066
(P. J. Bickel, 1975).

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 200_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---
# Bickel & Kim (2008), Monthly Weather Review 136(12).

# Tabela 3 do artigo: número de previsões para o MESMO DIA em cada valor anunciado.
PREVISTO = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
QUANTAS = np.array([4316, 2469, 4930, 2065, 799, 909, 602, 234, 606, 175, 233])
N_DIA = QUANTAS.sum()

# Tabela 2 do artigo, mesmo dia: média das previsões e frequência de chuva observada.
MEDIA_PREVISTA = 0.232
CHUVA_OBSERVADA = 0.180

# Texto do artigo: quando a previsão dizia 0,2 para o mesmo dia, choveu em 5,5% das vezes.
N_20 = 4930
FREQ_20 = 0.055

# Resumo do artigo: previsões entre 0,4 e 0,9 são bem calibradas no curto prazo.
# As frequências exatas por valor só aparecem em gráfico e não foram lidas.

# Tabela 2 do artigo: média anunciada menos chuva observada, por dia de antecedência (0 a 9).
DESVIO = np.array([0.052, 0.034, 0.031, 0.031, 0.025, 0.021, 0.019, -0.015, 0.052, 0.021])

# Tabela 4 do artigo: ganho sobre a média histórica (skill score, em %), por dia de antecedência.
ANTECEDENCIA = np.arange(10)
GANHO = np.array([35.9, 32.4, 24.2, 16.1, 11.8, 8.1, 5.4, 0.4, -12.0, -8.2])

N_TOTAL = 169163

print("=" * 70)
print("  DADOS · BICKEL & KIM (2008)")
print("=" * 70)
print(f"\n  Previsões no estudo: {mil(N_TOTAL)} · para o mesmo dia: {mil(N_DIA)}")
print(f"  Média das previsões do mesmo dia: {vir(MEDIA_PREVISTA * 100)}% · chuva observada: "
      f"{vir(CHUVA_OBSERVADA * 100)}%")
media_tabela = (PREVISTO * QUANTAS).sum() / N_DIA
print(f"  Conferência: média calculada pelas contagens da Tabela 3 = {vir(media_tabela * 100)}%")
print(f"\n  Valor anunciado e número de previsões (mesmo dia):")
for p, q in zip(PREVISTO, QUANTAS):
    print(f"    {p * 100:>4.0f}%  {mil(q):>6}  ({vir(q / N_DIA * 100)}% das previsões)")
print(f"\n  Previsões de 20%: {mil(N_20)} · choveu em {vir(FREQ_20 * 100)}%")
print("=" * 70)

# --- DADO BRASILEIRO · INMET, Bauru ---
# Arquivo dia-118-inmet-bauru-diario.csv, uma linha por dia com medição válida.
#   horas_aviso: horas do dia (0 a 24) com aviso de chuva do INMET em vigor para Bauru
#                (Chuvas Intensas, Tempestade ou Acumulado de Chuva, não cancelado)
#   horas_aviso_tempestade: o mesmo, só para avisos do tipo Tempestade
#   horas_aviso_laranja_vermelho: o mesmo, só para avisos de grau laranja ou vermelho
#   chuva_mm: chuva somada no dia (0h às 24h, horário de Brasília) na estação A705
# Ficaram fora os dias de 12/10/2025 a 17/03/2026. De janeiro a meados de março de 2026
# a estação não registrou chuva (campo vazio). De 12/10 a 31/12/2025 o registro tem
# sequências de dezenas de horas seguidas com 0,2 mm decrescente, padrão de pluviômetro
# entupido, e foi descartado por decisão minha.
with open('dia-118-inmet-bauru-diario.csv', encoding='utf-8') as arq:
    BAURU = list(csv.DictReader(arq))
HORAS_AVISO = np.array([int(r['horas_aviso']) for r in BAURU])
HORAS_TEMPESTADE = np.array([int(r['horas_aviso_tempestade']) for r in BAURU])
CHUVA_MM = np.array([float(r['chuva_mm']) for r in BAURU])
N_BAURU = len(BAURU)

print("\n" + "=" * 70)
print("  DADOS · INMET, BAURU (avisos e estação A705)")
print("=" * 70)
print(f"\n  Dias com medição válida: {N_BAURU} · de {BAURU[0]['data']} a {BAURU[-1]['data']}")
print(f"  Dias com alguma hora de aviso de chuva em vigor: {(HORAS_AVISO > 0).sum()}")
print(f"  Dias com chuva medida (qualquer quantidade): {(CHUVA_MM > 0).sum()}")
print(f"  Chuva somada no período: {vir(CHUVA_MM.sum())} mm")
print("=" * 70)

In [ ]:
# --- O MODELO ---
rng = np.random.default_rng(SEMENTE)

# ── CAMADA 1 · o exagero medido ──
chuvas_20 = round(FREQ_20 * N_20)
def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

post = jeffreys(chuvas_20, N_20)
ic20 = post.interval(0.95)
print("=" * 70)
print("  CAMADA 1 · O EXAGERO MEDIDO")
print("=" * 70)
print(f"\n  Mesmo dia: a previsão anunciou em média {vir(MEDIA_PREVISTA * 100)}% · choveu em "
      f"{vir(CHUVA_OBSERVADA * 100)}% dos dias")
print(f"  Diferença: {vir((MEDIA_PREVISTA - CHUVA_OBSERVADA) * 100)} pontos percentuais · "
      f"razão {vir(MEDIA_PREVISTA / CHUVA_OBSERVADA, 2)}")
print(f"  Dias de antecedência em que a média anunciada ficou acima da chuva observada: "
      f"{(DESVIO > 0).sum()} de {len(DESVIO)}")
print(f"  Previsões de 20% entre as do mesmo dia: {vir(N_20 / N_DIA * 100)}% (o valor mais usado)")
print("\n  Quando a previsão diz 20%:")
print(f"\n  {mil(N_20)} previsões de 20% · choveu em cerca de {chuvas_20} ({vir(FREQ_20 * 100)}%)")
print(f"  Posterior de Jeffreys: {vir(post.mean() * 100)}% · IC 95% [{vir(ic20[0] * 100)}; {vir(ic20[1] * 100)}]")
print(f"  A previsão anunciou {vir(0.20 / FREQ_20)} vezes mais chuva do que houve")
print("\n  Sensibilidade: previsões da mesma cidade em dias seguidos se parecem.")
print("  Se cada previsão valer menos do que uma observação independente:")
sens = {}
for fator in (1, 5, 10, 20):
    n_ef = N_20 / fator
    d = jeffreys(FREQ_20 * n_ef, n_ef)
    lo, hi = d.interval(0.95)
    sens[fator] = (lo, hi, 1 - d.cdf(0.20))
    print(f"    amostra efetiva de {mil(n_ef):>5}: IC 95% [{vir(lo * 100)}; {vir(hi * 100)}] · "
          f"P(frequência real ≥ 20%) menor que 0,001%")
    assert sens[fator][2] < 1e-5

# ── CAMADA 2 · cem dias de 20% ──
p_sorteado = post.rvs(SORTEIOS, random_state=rng)
dias_chuva = rng.binomial(100, p_sorteado)
dias_prometidos = rng.binomial(100, 0.20, size=SORTEIOS)
dc_lo, dc_hi = np.percentile(dias_chuva, [2.5, 97.5])
print("\n" + "=" * 70)
print(f"  CAMADA 2 · CEM DIAS COM PREVISÃO DE 20% ({mil(SORTEIOS)} sorteios)")
print("=" * 70)
print(f"\n  Dias de chuva, pelo que foi medido: média {vir(dias_chuva.mean())} · "
      f"95% entre {int(dc_lo)} e {int(dc_hi)}")
print(f"  Dias de chuva, se os 20% valessem: média {vir(dias_prometidos.mean())}")
print(f"  P(chover em 20 dias ou mais, pelo medido): {vir((dias_chuva >= 20).mean() * 100, 3)}%")
print(f"  Dias secos em cem: {vir(100 - dias_chuva.mean())}")

# ── CAMADA 3 · a antecedência ──
ate7 = ANTECEDENCIA <= 7
reta = stats.linregress(ANTECEDENCIA[ate7], GANHO[ate7])
dia_zero = -reta.intercept / reta.slope
print("\n" + "=" * 70)
print("  CAMADA 3 · O GANHO SOBRE A MÉDIA HISTÓRICA, POR ANTECEDÊNCIA")
print("=" * 70)
for d, g in zip(ANTECEDENCIA, GANHO):
    print(f"    {d} dia(s) antes: {vir(g):>6}%")
print(f"\n  Reta ajustada (0 a 7 dias): ganho = {vir(reta.intercept)} − {vir(-reta.slope)} × dias "
      f"· R² = {vir(reta.rvalue ** 2, 3)}")
print(f"  A reta chega a zero em {vir(dia_zero)} dias")
print(f"  Com 8 e 9 dias o ganho é negativo: a média histórica prevê melhor")
# ── CAMADA 4 · e ela acerta? acertar no sentido comum (chove ou não chove, corte em 50%) ──
# Identidade usada: o total de dias com chuva é conhecido (18,0% de 17.338).
# Só é preciso saber quantos desses dias caíram do lado "vai chover" (previsão >= 50%).
#   acertos = secos previstos e confirmados + chuvas previstas e confirmadas
#           = [N_seco − (chuva_total − chuva_no_lado_molhado)] + chuva_no_lado_molhado
# Premissas, declaradas:
#   a) nos valores de 0,5 a 0,9 a previsão é bem calibrada (o artigo afirma isso);
#      a frequência real de cada valor é sorteada em previsto ± 5 pontos;
#   b) no valor 1,0 o artigo diz que não há boa calibração; frequência sorteada entre 70% e 100%;
#   c) a chuva observada de 18,0% é arredondada; sorteada entre 17,95% e 18,05%.
molhado = PREVISTO >= 0.5
N_SECO = QUANTAS[~molhado].sum()
N_MOLHADO = QUANTAS[molhado].sum()

def acerto(freq_molhado, chuva_obs):
    chuva_total = chuva_obs * N_DIA
    chuva_molhado = (QUANTAS[molhado] * freq_molhado).sum(axis=-1)
    return (N_SECO - (chuva_total - chuva_molhado) + chuva_molhado) / N_DIA

centro = PREVISTO[molhado].copy()
centro[-1] = 0.85
acerto_central = acerto(centro, CHUVA_OBSERVADA)

ruido = rng.uniform(-0.05, 0.05, size=(SORTEIOS, molhado.sum()))
freqs = PREVISTO[molhado] + ruido
freqs[:, -1] = rng.uniform(0.70, 1.00, size=SORTEIOS)
obs = rng.uniform(0.1795, 0.1805, size=SORTEIOS)
acertos = acerto(freqs, obs)
ac_lo, ac_med, ac_hi = np.percentile(acertos, [2.5, 50, 97.5])
ingenuo = 1 - CHUVA_OBSERVADA
ganho_pontos = (acertos - ingenuo) * 100

print("\n" + "=" * 70)
print("  CAMADA 4 · CHOVE OU NÃO CHOVE, COM CORTE EM 50%")
print("=" * 70)
print(f"\n  Previsões abaixo de 50%: {mil(N_SECO)} ({vir(N_SECO / N_DIA * 100)}%) · "
      f"de 50% para cima: {mil(N_MOLHADO)} ({vir(N_MOLHADO / N_DIA * 100)}%)")
print(f"  Taxa de acerto reconstruída (valor central): {vir(acerto_central * 100)}%")
print(f"  Com as premissas sorteadas {mil(SORTEIOS)} vezes: mediana {vir(ac_med * 100)}% · "
      f"95% entre {vir(ac_lo * 100)}% e {vir(ac_hi * 100)}%")
print(f"\n  Quem dissesse 'não vai chover' todos os dias acertaria {vir(ingenuo * 100)}%")
print(f"  Vantagem da previsão: {vir(np.median(ganho_pontos))} pontos percentuais "
      f"(95% entre {vir(np.percentile(ganho_pontos, 2.5))} e {vir(np.percentile(ganho_pontos, 97.5))})")
print(f"  P(a previsão ganha de 'nunca chove'): {vir((ganho_pontos > 0).mean() * 100)}%")

# Conferência com o erro quadrático publicado (0,095), usando uma alocação ilustrativa
# para os valores não lidos: 0,3 igual à média histórica e 0,0 e 0,1 dividindo o que sobra.
freq_ilus = PREVISTO.copy()
freq_ilus[2], freq_ilus[3], freq_ilus[10] = FREQ_20, CHUVA_OBSERVADA, 0.85
resto = CHUVA_OBSERVADA * N_DIA - (QUANTAS * freq_ilus)[2:].sum()
freq_ilus[0] = freq_ilus[1] = resto / (QUANTAS[0] + QUANTAS[1])
eqm = (QUANTAS * (freq_ilus * (1 - PREVISTO) ** 2 + (1 - freq_ilus) * PREVISTO ** 2)).sum() / N_DIA
print(f"\n  Conferência: erro quadrático médio da reconstrução = {vir(eqm, 3)} · publicado = 0,095")
print(f"  Frequência de chuva que sobra para os valores 0% e 10%: {vir(freq_ilus[0] * 100)}%")
print("=" * 70)

# ── CAMADA 5 · o dado brasileiro: dias com aviso do INMET em Bauru ──
# Dia com aviso: aviso de chuva em vigor por 12 horas ou mais naquele dia.
# Choveu: qualquer quantidade registrada na estação (a menor medida é 0,2 mm).
com_aviso = HORAS_AVISO >= 12
n_av, k_av = int(com_aviso.sum()), int((CHUVA_MM[com_aviso] > 0).sum())
n_sem, k_sem = int((~com_aviso).sum()), int((CHUVA_MM[~com_aviso] > 0).sum())
post_av, post_sem = jeffreys(k_av, n_av), jeffreys(k_sem, n_sem)
ic_av, ic_sem = post_av.interval(0.95), post_sem.interval(0.95)
razao = post_av.rvs(SORTEIOS, random_state=rng) / post_sem.rvs(SORTEIOS, random_state=rng)
rz_lo, rz_med, rz_hi = np.percentile(razao, [2.5, 50, 97.5])

print("\n" + "=" * 70)
print("  CAMADA 5 · BAURU: DIAS COM AVISO DE CHUVA DO INMET")
print("=" * 70)
print(f"\n  Dias com aviso: {n_av} ({vir(n_av / N_BAURU * 100)}% dos {N_BAURU} dias)")
print(f"    choveu em {k_av} · {vir(k_av / n_av * 100)}% · IC 95% [{vir(ic_av[0] * 100)}; {vir(ic_av[1] * 100)}]")
print(f"    não choveu na estação em {n_av - k_av} ({vir((n_av - k_av) / n_av * 100)}%)")
print(f"  Dias sem aviso: {n_sem}")
print(f"    choveu em {k_sem} · {vir(k_sem / n_sem * 100)}% · IC 95% [{vir(ic_sem[0] * 100)}; {vir(ic_sem[1] * 100)}]")
print(f"  Razão entre as duas frequências: {vir(rz_med)} · 95% entre {vir(rz_lo)} e {vir(rz_hi)}")
print(f"\n  Volume: {vir(CHUVA_MM[com_aviso].sum())} mm caíram em dias com aviso, "
      f"{vir(CHUVA_MM[com_aviso].sum() / CHUVA_MM.sum() * 100)}% da chuva do período")
print(f"  Maior chuva em um dia com aviso: {vir(CHUVA_MM[com_aviso].max())} mm")
print("\n  Por quantidade de chuva, nos dias com aviso e sem aviso:")
for corte in (1, 10):
    a, b = int((CHUVA_MM[com_aviso] >= corte).sum()), int((CHUVA_MM[~com_aviso] >= corte).sum())
    print(f"    {corte:>2} mm ou mais: {a} de {n_av} ({vir(a / n_av * 100)}%) · "
          f"{b} de {n_sem} ({vir(b / n_sem * 100)}%)")
print("\n  Outras formas de contar:")
temp = HORAS_TEMPESTADE >= 12
k_t, n_t = int((CHUVA_MM[temp] > 0).sum()), int(temp.sum())
ic_t = jeffreys(k_t, n_t).interval(0.95)
print(f"    só avisos do tipo Tempestade: choveu em {k_t} de {n_t} dias · {vir(k_t / n_t * 100)}% · "
      f"IC 95% [{vir(ic_t[0] * 100)}; {vir(ic_t[1] * 100)}]")
qualquer = HORAS_AVISO >= 1
k_q, n_q = int((CHUVA_MM[qualquer] > 0).sum()), int(qualquer.sum())
print(f"    aviso em vigor por 1 hora ou mais no dia: choveu em {k_q} de {n_q} dias · {vir(k_q / n_q * 100)}%")
nenhum = HORAS_AVISO == 0
k_n, n_n = int((CHUVA_MM[nenhum] > 0).sum()), int(nenhum.sum())
print(f"    dias sem nenhuma hora de aviso: choveu em {k_n} de {n_n} · {vir(k_n / n_n * 100)}%")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fonte: Bickel & Kim (2008), Monthly Weather Review 136(12) · N = 169.163 previsões | #365Probabilidades'

# ══ GRÁFICO 1 · os 20% que não são 20% ══
fig1, ax1 = plt.subplots(figsize=(12, 8))
x = np.linspace(0.03, 0.22, 800)
ax1.plot(x * 100, post.pdf(x), color=VERMELHO, linewidth=3)
ax1.fill_between(x * 100, post.pdf(x), alpha=0.2, color=VERMELHO)
ax1.axvline(ic20[0] * 100, color=DOURADO, linestyle='--', linewidth=2)
ax1.axvline(ic20[1] * 100, color=DOURADO, linestyle='--', linewidth=2)
ax1.axvline(20, color=CINZA, linewidth=2.5)
ax1.text(19.7, post.pdf(post.mean()) * 0.55, 'o que a previsão\nanunciou: 20%', ha='right',
         fontsize=11, color=CINZA)
ax1.text(ic20[1] * 100 + 0.3, post.pdf(post.mean()) * 0.85,
         f'o que choveu: {vir(FREQ_20 * 100)}%\nIC 95% [{vir(ic20[0] * 100)}; {vir(ic20[1] * 100)}]',
         fontsize=11, color=VERMELHO)
ax1.set_xlabel('Frequência de chuva nos dias com previsão de 20% (%)', fontsize=12)
ax1.set_ylabel('Densidade', fontsize=12)
ax1.set_title(f'Distribuição Beta (Jeffreys) · {mil(N_20)} previsões de 20% para o mesmo dia',
              fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-118-grafico-01-vinte-por-cento.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · cem dias de 20% ══
fig2, ax2 = plt.subplots(figsize=(12, 8))
k = np.arange(0, 41)
ax2.bar(k - 0.2, np.bincount(dias_chuva, minlength=41)[:41] / SORTEIOS * 100, width=0.4,
        color=VERMELHO, alpha=0.85, label=f'pelo que foi medido (média {vir(dias_chuva.mean())})')
ax2.bar(k + 0.2, np.bincount(dias_prometidos, minlength=41)[:41] / SORTEIOS * 100, width=0.4,
        color=CINZA, alpha=0.7, label=f'se os 20% valessem (média {vir(dias_prometidos.mean())})')
ax2.set_xlabel('Dias de chuva em cem dias com previsão de 20%', fontsize=12)
ax2.set_ylabel('Sorteios (%)', fontsize=12)
ax2.set_title('Cem dias em que a previsão disse 20%\n'
              f'{mil(SORTEIOS)} sorteios, semente {SEMENTE}', fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=11)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-118-grafico-02-cem-dias.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · a antecedência ══
fig3, ax3 = plt.subplots(figsize=(12, 8))
cores = [VERDE if g > 0 else VERMELHO for g in GANHO]
ax3.bar(ANTECEDENCIA, GANHO, color=cores, alpha=0.85)
for d, g in zip(ANTECEDENCIA, GANHO):
    ax3.text(d, g + (1.2 if g >= 0 else -2.6), vir(g), ha='center', fontsize=10)
xs = np.linspace(0, 7, 50)
ax3.plot(xs, reta.intercept + reta.slope * xs, color=DOURADO, linewidth=2.5, linestyle='--',
         label=f'reta: {vir(reta.intercept)} − {vir(-reta.slope)} × dias')
ax3.axhline(0, color=CINZA, linewidth=1.2)
ax3.set_xticks(ANTECEDENCIA)
ax3.set_ylim(-18, 42)
ax3.set_xlabel('Dias de antecedência', fontsize=12)
ax3.set_ylabel('Ganho sobre a média histórica (%)', fontsize=12)
ax3.set_title('Quanto a previsão vale a cada dia de antecedência\n'
              'depois de sete dias, a média histórica prevê melhor', fontsize=14, pad=16)
ax3.legend(frameon=False, fontsize=11)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-118-grafico-03-antecedencia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · acerto reconstruído contra "nunca chove" ══
fig4, ax4 = plt.subplots(figsize=(12, 8))
ax4.hist(acertos * 100, bins=60, color=VERDE, alpha=0.8)
ax4.axvline(ingenuo * 100, color=VERMELHO, linewidth=2.5)
ax4.text(ingenuo * 100 + 0.1, ax4.get_ylim()[1] * 0.92, f"'não vai chover' todo dia\n{vir(ingenuo * 100)}%",
         color=VERMELHO, fontsize=11)
ax4.axvline(ac_med * 100, color=DOURADO, linewidth=2, linestyle='--')
ax4.text(ac_med * 100 + 0.1, ax4.get_ylim()[1] * 0.75, f"previsão\nmediana {vir(ac_med * 100)}%",
         color=DOURADO, fontsize=11)
ax4.set_xlim(81, 90)
ax4.set_xlabel('Dias em que a previsão acertou chuva ou não chuva (%)', fontsize=12)
ax4.set_ylabel('Sorteios', fontsize=12)
ax4.set_title('A previsão do mesmo dia acerta quase nove em cada dez\n'
              'reconstrução com corte em 50%, premissas sorteadas 200 mil vezes', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-118-grafico-04-acerto.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# ══ GRÁFICO 5 · Bauru: dias com aviso e dias sem aviso ══
fig5, ax5 = plt.subplots(figsize=(12, 8))
rotulos = [f'Dias com aviso de chuva\n({n_av} dias)', f'Dias sem aviso\n({n_sem} dias)']
valores = np.array([k_av / n_av, k_sem / n_sem]) * 100
erros = np.array([[valores[0] - ic_av[0] * 100, valores[1] - ic_sem[0] * 100],
                  [ic_av[1] * 100 - valores[0], ic_sem[1] * 100 - valores[1]]])
barras = ax5.bar(rotulos, valores, color=[VERDE, CINZA], alpha=0.85, width=0.5,
                 yerr=erros, capsize=8, ecolor=DOURADO, error_kw={'linewidth': 2.5})
for b, v, ic in zip(barras, valores, (ic_av, ic_sem)):
    ax5.text(b.get_x() + b.get_width() / 2, ic[1] * 100 + 2.5,
             f'{vir(v)}%\nIC 95% [{vir(ic[0] * 100)}; {vir(ic[1] * 100)}]',
             ha='center', fontsize=13, fontweight='bold')
ax5.set_ylim(0, 85)
ax5.set_ylabel('Dias em que choveu na estação (%)', fontsize=12)
ax5.set_title('Bauru: choveu nos dias em que havia aviso de chuva do INMET?\n'
              f'{N_BAURU} dias com medição válida, de março de 2025 a agosto de 2026', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, 'Fonte: INMET, avisos meteorológicos e estação automática A705 (Bauru) · '
            'cálculo próprio | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-118-grafico-05-bauru.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

### 💡 O Insight

**Quando a previsão dizia 20% de chance de chuva para o mesmo dia, choveu em 5,5% das vezes.**

Foram 4.930 previsões desse tipo, o valor mais usado pelo serviço (28,4% das previsões
do mesmo dia). O intervalo da frequência real vai de 4,9% a 6,2%. De cada cem dias com
esse aviso, a simulação dá entre 2 e 10 dias de chuva, e não 20.

O exagero aparecia nos números pequenos. O artigo explica que o sistema tratava 29% como
o limite de uma previsão seca, e os valores abaixo disso saíam inflados. Entre 40% e 90%
a previsão de curto prazo correspondia ao que acontecia. No conjunto, a previsão do mesmo
dia anunciava em média 23,2% de chance de chuva, e choveu em 18,0% dos dias.

A antecedência também pesa. No mesmo dia a previsão era 35,9% melhor que a média
histórica. Com seis dias, 5,4%. Com sete, 0,4%. Com oito e nove dias a média histórica
previa melhor que a previsão.

Exagerar nos valores baixos não quer dizer errar muito. Com corte em 50%, a previsão do
mesmo dia acertou chuva ou não chuva em cerca de 87% das vezes, pela minha reconstrução.
Quem dissesse "não vai chover" todos os dias acertaria 82,0%.

**Em Bauru, choveu em 56,6% dos dias com aviso de chuva do INMET e em 10,9% dos dias sem
aviso.** São 56 de 99 dias contra 32 de 293, entre março de 2025 e agosto de 2026. Com
aviso do tipo Tempestade, choveu em 32 de 44 dias (72,7%). Os dias com aviso são 25,3%
do período e receberam 86,9% da chuva medida. Em 43 dos 99 dias com aviso não caiu chuva
na estação.

Para o treino, fico com três leituras. Se a previsão de hoje mostra uma chance pequena
de chuva, a chance real é menor ainda. A previsão para daqui a uma semana não ajuda a
decidir. E em dia de aviso de chuva para Bauru, choveu em pouco mais da metade das vezes.

*Quando a previsão diz 20%, você leva o guarda-chuva?*

---

### ⚠️ Limitações do Modelo

- **As camadas 1 a 4 são de um serviço, um país, uma época.** Os dados são do The Weather
  Channel, em 42 cidades dos Estados Unidos, entre 2004 e 2006. Não medem os aplicativos
  usados hoje no Brasil.
- **Aviso do INMET não é probabilidade de chuva.** É um aviso de risco para uma região.
  A camada 5 mede se choveu nos dias com aviso. Ela não mede calibração, e os 43 dias
  com aviso e sem chuva na estação não são, sozinhos, prova de exagero.
- **O aviso cobre uma área e a estação mede um ponto.** Os avisos que incluíam Bauru
  listavam de 159 a 5.051 municípios (mediana de 1.718). Pode ter chovido em outra parte
  da cidade sem registro na estação A705.
- **Faltam os meses mais chuvosos.** Os dias de 12/10/2025 a 17/03/2026 ficaram fora.
  De janeiro a meados de março de 2026 a estação não registrou chuva. De 12/10 a
  31/12/2025 descartei o registro por decisão minha, porque ele mostra dezenas de horas
  seguidas de 0,2 mm decrescente, padrão de pluviômetro entupido. O resultado vale para
  392 dias de outono, inverno e começo da primavera.
- **A regra de "dia com aviso" é minha.** Contei o dia quando havia aviso em vigor por
  12 horas ou mais. Contando qualquer dia com 1 hora de aviso, choveu em 65 de 129
  (50,4%). O horário de cancelamento dos avisos não é publicado, e os avisos que
  constam como cancelados ficaram fora.
- **A tabela diária de Bauru foi montada a partir dos arquivos do INMET fora deste
  notebook.** O notebook lê o arquivo dia-118-inmet-bauru-diario.csv. Cinco IDs de aviso
  retornaram erro do servidor e não foram lidos.
- **Dias seguidos se parecem.** Vale para as previsões americanas e para os dias de
  Bauru. Os intervalos tratam cada dia como independente e por isso são estreitos demais.
  Nos 20%, com a amostra efetiva reduzida em até vinte vezes, o intervalo vai de 3,2% a
  8,9% e continua longe de 20%.
- **O exagero foi lido com número só nos 20%.** A frequência de chuva para os outros
  valores anunciados aparece apenas em gráfico no artigo e não foi lida. O número de
  dias com chuva nas previsões de 20% foi recuperado do percentual (5,5% de 4.930 dá
  cerca de 271).
- **Chuva, no estudo americano, é qualquer registro a partir de 0,01 polegada entre 7h
  e 19h.** Em Bauru é qualquer registro a partir de 0,2 mm entre 0h e 24h. Garoa e
  temporal contam igual.
- **A taxa de acerto é reconstrução nossa.** O artigo não publica acerto com corte em
  50%. Ela sai do total de dias com chuva, das contagens por valor anunciado e da
  afirmação dos autores de que os valores de 40% a 90% são bem calibrados. O erro
  quadrático médio da reconstrução é 0,103, contra 0,095 publicado, então o acerto
  verdadeiro pode ser um pouco maior. Com outro corte o acerto muda.
- **A reta da antecedência é descritiva.** Oito pontos, sem modelo físico por trás.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*